# Quickstart: inspect a reason-deletion certificate

> **Demonstration only:** frozen synthetic data, not evidence about any real decision. Reasonsmith reports evidence and refusals; it does not certify compliance or provide legal advice.

The next cell uses the library API to run the shipped credit system and keeps the report object. The first table is the payoff: one decision stated one reason, inference found five, and the deletion measurement identified the four omitted dependencies.

In [1]:
import html
from dataclasses import replace

from IPython.display import HTML, display

from reasonsmith.demo import deployed_credit_system
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack


def show_table(rows, columns, title=None):
    heading = f"<h3>{html.escape(title)}</h3>" if title else ""
    head = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(column, '—')))}</td>" for column in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"{heading}<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

pack = load_pack("ecoa")
requirement = pack.get_requirement("ecoa_reg_b_1002_9_b_2_principal_reasons_complete")
one_duty = replace(pack, id="ecoa:principal-reasons", requirements=(requirement,))
report = check_conformance(deployed_credit_system(), one_duty)
result = report.to_dict()["results"][0]
certificate = result["details"]["certificates"][1]
rows = [{"Decision": "APP-1042", "Stated": 1, "Used": certificate["reasons_found"],
         "Measured": "deleted", "Reason": reason}
        for reason in certificate["missing_reasons"]]
show_table([{"Duty": result["requirement_id"], "Verdict": result["verdict"],
            "Rung": result["strength"], "Basis": result["basis"],
            "Witness / refusal": certificate["attribution"]}],
           ["Duty", "Verdict", "Rung", "Basis", "Witness / refusal"],
           "Result")
show_table(rows, ["Decision", "Stated", "Used", "Measured", "Reason"],
           "The four reasons that made no difference")


Duty,Verdict,Rung,Basis,Witness / refusal
ecoa_reg_b_1002_9_b_2_principal_reasons_complete,violated,probed,artifact,"The deleted reasons are exactly the 4 lowest-scoring of the 5, and the engine kept the top 1. This is the signature of top-k proof truncation at k=1: top-k works by discarding proofs, so the dropped reasons are lost by configuration, not by error. The missing probability mass is 0.225799."


Decision,Stated,Used,Measured,Reason
APP-1042,1,5,deleted,C05 — Insufficient number of credit references provided
APP-1042,1,5,deleted,C03 — Delinquent past or present credit obligations
APP-1042,1,5,deleted,C04 — Too many recent inquiries on credit bureau report
APP-1042,1,5,deleted,C02 — Length of time credit has been established is too short


The certificate is inspectable structure, not a log-reading heuristic. Reasonsmith enumerates the inference artefact, switches off each reason's private facts, and reruns the system; a reason is deleted when that counterfactual leaves the answer unchanged. The log's single statement is only the comparison target. This is the deletion contract in [`docs/theory/07-explanation.md`](../docs/theory/07-explanation.md).

In [2]:
show_table([{"Duty": result["requirement_id"], "Verdict": result["verdict"],
             "Rung": result["strength"], "Basis": result["basis"],
             "Witness / refusal": certificate["attribution"],
             "Probe trials": result["details"]["probe_budget"]["trials"]}],
            ["Duty", "Verdict", "Rung", "Basis", "Witness / refusal", "Probe trials"],
            "Certificate details")

Duty,Verdict,Rung,Basis,Witness / refusal,Probe trials
ecoa_reg_b_1002_9_b_2_principal_reasons_complete,violated,probed,artifact,"The deleted reasons are exactly the 4 lowest-scoring of the 5, and the engine kept the top 1. This is the signature of top-k proof truncation at k=1: top-k works by discarding proofs, so the dropped reasons are lost by configuration, not by error. The missing probability mass is 0.225799.",13


For comparison, the same frozen run is available from the shell. The CLI is an export of the same result object, not the notebook's implementation path.

In [3]:
import subprocess

shell = subprocess.run([
    "reasonsmith", "check", "--system-module",
    "reasonsmith.examples.truncating_credit_system:system_under_test", "--pack", "ecoa",
], check=False, capture_output=True, text=True)
print(f"CLI comparison: exit status {shell.returncode} (same result, rendered by shell)")

CLI comparison: exit status 2 (same result, rendered by shell)
